<a href="https://colab.research.google.com/github/Demosthene-OR/Student-AI-and-Data-Management/blob/main/data_80/maintenance_project/Predictive_Maintenance_Light_Starter_Guide.ipynb"
   target="_blank"
   rel="noopener noreferrer">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

<img src="https://prof.totalenergies.com/wp-content/uploads/2024/09/TotalEnergies_TPA_picto_DegradeRouge_RVB-1024x1024.png" height="150" width="150">
<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> 🔧 Predictive Maintenance Exercise - Oil and Gas Industry </h1></center>

<hr style="border-width:2px;border-color:#75DFC1">

## 📋 EXERCISE STATEMENT

### Context
You are a data scientist for an oil/gas company. One critical pieces of equipment regularly fail:
- **Centrifugal pumps** : Cost \$50k/day when shut down     

### Objective
Develop a **predictive maintenance** system to:
1. Predict the **Remaining Useful Life (RUL)** in cycles
2. Classify whether a **failure will occur in N cycles**
3.  Identify **critical sensors** (feature importance)

### Data
- **Dataset** : `pump_predictive_maintenance.csv` (5162 measurements, 30 pumps)
-  Type: Run-to-failure data (from start-up to failure)

### Pump dataset  

| Variable | Description |
| :-- | :-- |
| engine_id | Unique identifier for each pump unit. Each pump has multiple records representing its operating history from startup until failure. |
| cycle | Operating cycle number (time step) for the pump. Cycle 0 is the first measurement, and each pump's last cycle represents the point of failure (RUL=0). |
| vibration_mm_s | Vibration amplitude in millimeters per second (mm/s). Measures mechanical oscillations of the pump. Higher vibration typically indicates bearing wear, imbalance, or cavitation. Strong predictor of pump degradation. |
| temperature_C | Fluid temperature in degrees Celsius. Indicates heat generation inside the pump. Elevated temperature suggests increased friction, wear, or operating stress. Includes seasonal variations (day/night cycles). |
| pressure_bar | Discharge pressure in bar (absolute pressure unit). Measures the pressure the pump can generate. Declining pressure over time suggests internal leakage or loss of pump efficiency due to wear. |
| flow_rate_m3h | Volume flow rate in cubic meters per hour (m³/h). Measures the volume of fluid pumped per unit time. Decreases with wear and degradation. Strongly influenced by operating load and external demand (seasonality). |
| motor_current_A | Electrical current drawn by the motor in amperes (A). Indicates motor load and power consumption. Increases with friction and mechanical resistance as the pump degrades. Useful for detecting efficiency loss. |
| aux_sensor_unitless | Auxiliary sensor reading (unitless). Simulates environmental or contextual measurements (e.g., ambient temperature, barometric pressure, or facility conditions). Weakly correlated with pump RUL—useful for demonstrating feature selection and the difference between informative vs. noise signals. |
| rul_cycles | Remaining Useful Life in cycles until failure. Decreases monotonically from the pump’s maximum life to 0 at failure. Perfect run-to-failure label: each pump reaches RUL=0 on its last recorded cycle. Used as the regression target to predict remaining lifespan. |



## 📦 Part 1: Load Libraries & Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, accuracy_score, confusion_matrix, mean_absolute_error, r2_score

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load pump dataset
url = "https://raw.githubusercontent.com/Demosthene-OR/Student-AI-and-Data-Management/main/data_80/maintenance_project/"
pump_df = pd.read_csv(url+'pump_predictive_maintenance.csv')

# Insert your code here

## ✅ Part 2: Sanity Checks (Run-to-Failure)

Verify that each machine runs until RUL = 0 (failure).

Hint Use `groupby()` to group by machine ID, then check `min(RUL)` for each machine.  
Expected result : All machines should have min(RUL) = 0 (they all run until failure)

In [ ]:
# Insert your code here


## 📊 Part 3 : Exploratory Data Analysis (EDA) – Feature Correlation Analysis

### Key Insight
In real datasets, NOT all sensors correlate equally with RUL.
- Some sensors show clear degradation trends → useful for prediction
- Others are noisy, seasonal, or uncorrelated → less useful

We'll display pumps sensors evolution and distribution, then compute correlations and create a heatmap.

### 3.1 - Sensor Correlation with RUL

### 🎯 TODO: Calculate correlations

**Hint**: Use `.corr()` method to calculate correlation matrix. Then extract correlations with RUL column.

In [ ]:
# Insert your code here


### 3.2 - Correlation Matrix Heatmap

### 🎯 TODO: Create a heatmap for pumps

**Hint**: Use `sns.heatmap()` with correlation matrix. Parameter: `annot=True` to show values.

In [ ]:
# Insert your code here


### 3.3 - Sensor Evolution for One Machine

### 🎯 TODO: Visualize how one pump degrades over time

**Hint**: Filter data for one or few machine(s) (engine_id=x), then plot each sensor vs cycle.

In [ ]:
# Insert your code here


## 🔨 Part 4: Feature Engineering

### 🎯 TODO: Create advanced features (Optional)

**Features to create (Optional):**
1. Rolling mean (smoothing) - window size 3 and 5
2. Rolling std (variability) - window size 3 and 5
3. Gradient (rate of change)

**Hint**: Use `.groupby('engine_id').transform()` to operate on each machine separately

In [ ]:
# Insert your code here


## 📋 Part 5: Target Creation & Data Preparation

### 🎯 TODO: Create targets for prediction, create train & test datasets, normalize features

##### **Two approaches:**
1. **Regression target**: RUL itself (rul_cycles)
2. **Classification target**: Binary (will fail in next N cycles?)

##### **Hint (important): use a time-aware split (no leakage) :**

Even if you only have **one run-to-failure history per pump**, avoid a random row-wise split: it would mix early and late cycles of the *same* pump in both train and test, creating **temporal leakage** and overly optimistic scores.
A simple and robust alternative is a **pump-level split** with `GroupShuffleSplit(groups=engineid)`[*(see documentation here)*](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html) : it keeps entire pumps either in train or in test, so the model is evaluated on **unseen pumps** (a realistic generalization check).
`train_idx, test_idx = next(gss.split(X, y, groups))` just takes the first (and here only) split produced by the generator returned by `.split(...)`, and you use those indices to slice `X`/`y` into train and test. 

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Insert your code here


## 🤖 Part 6: Model Training

### 🎯 TODO: Train models

**Tasks:**
1. Choose models for RUL Regressor & Failure Classifier
2. Train RUL Regressor
3. Train Failure Classifier
4. Calculate metrics for each model

In [ ]:
# Insert your code here


In [ ]:
# Insert your code here


## 📈 Part 7: Visualize Results

### 🎯 TODO: Create visualizations

**Charts:** for example,  
1. Predictions vs Reality (scatter plot)
2. Feature Importance (bar plot)

In [ ]:
# Insert your code here


In [ ]:
# Insert your code here


## 🎓 Part 8: Summary & Insights

### 🎯 TODO: Write your conclusions

**Questions to answer:**
1. Which sensor is most useful for RUL prediction?
2. What is the best model performance (R² for regression)?
3. Can we detect failures before they happen (classification)?
4. What would you recommend for production deployment?

In [ ]:
# Insert your code here

print("\n" + "="*70)
print("CONCLUSIONS\n" + "="*70)
print("""
1. BEST SENSORS FOR PREDICTION:


2. MODEL PERFORMANCE:

   
3. FAILURE DETECTION:


4. PRODUCTION RECOMMENDATIONS:

""")

## 🎉 CONGRATULATIONS! 🎉

You've completed a full predictive maintenance project!

**Skills learned:**
✅ Data exploration & analysis
✅ Feature engineering
✅ Model training & evaluation
✅ Regression vs Classification
✅ Machine Learning fundamentals

**Next steps:**
→ Try different hyperparameters
→ Compare with other models (Gradient Boosting, Neural Networks)
→ Implement cross-validation
→ Optimize for production

---